# E-commerce Clickstream Data Pipeline

This notebook builds a simple medallion-style pipeline in Databricks using PySpark, Delta tables, and SQL.

**Flow:** Raw CSV → Bronze → Silver → Gold → Visualizations


## 1. Bronze Layer - Raw Data Ingestion

Load the sampled e-commerce CSV from a Unity Catalog volume and persist the raw data as a Bronze Delta table without changing the source structure.


In [0]:
raw_df = spark.read.option("header", True).csv(
    "/Volumes/workspace/bronze/raw_data/ecommerce_sample.csv"
)

display(raw_df)

In [0]:
raw_df.printSchema()

In [0]:
raw_df.write.mode("overwrite").saveAsTable(
    "workspace.bronze.ecommerce_events"
)

In [0]:
%sql
SELECT COUNT(*) 
FROM workspace.bronze.ecommerce_events;

In [0]:
%sql
SELECT *
FROM workspace.bronze.ecommerce_events
LIMIT 10;

## 2. Data Quality Checks

Inspect event distribution, missing values, and duplicate records before applying any cleaning rules.


In [0]:
%sql
SELECT event_type, COUNT(*) AS count
FROM workspace.bronze.ecommerce_events
GROUP BY event_type
ORDER BY count DESC;

In [0]:
%sql
SELECT
  SUM(CASE WHEN event_time IS NULL THEN 1 ELSE 0 END) AS missing_event_time,
  SUM(CASE WHEN event_type IS NULL THEN 1 ELSE 0 END) AS missing_event_type,
  SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS missing_product_id,
  SUM(CASE WHEN category_code IS NULL THEN 1 ELSE 0 END) AS missing_category_code,
  SUM(CASE WHEN brand IS NULL THEN 1 ELSE 0 END) AS missing_brand,
  SUM(CASE WHEN price IS NULL THEN 1 ELSE 0 END) AS missing_price,
  SUM(CASE WHEN user_id IS NULL THEN 1 ELSE 0 END) AS missing_user_id,
  SUM(CASE WHEN user_session IS NULL THEN 1 ELSE 0 END) AS missing_user_session
FROM workspace.bronze.ecommerce_events;

In [0]:
%sql
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT event_time, event_type, product_id, user_id, user_session) AS distinct_rows
FROM workspace.bronze.ecommerce_events;

In [0]:
%sql
SELECT
  event_time,
  event_type,
  product_id,
  user_id,
  user_session,
  COUNT(*) AS duplicate_count
FROM workspace.bronze.ecommerce_events
GROUP BY
  event_time,
  event_type,
  product_id,
  user_id,
  user_session
HAVING COUNT(*) > 1
ORDER BY duplicate_count DESC
LIMIT 10;

## 3. Silver Layer - Cleaning and Type Conversion

Remove duplicate events, convert important columns to proper data types, validate the converted values, and filter non-positive prices.


In [0]:
bronze_df = spark.table("workspace.bronze.ecommerce_events")

# Remove duplicate events using the fields that identify the same user action.
silver_df = bronze_df.dropDuplicates([
    "event_time",
    "event_type",
    "product_id",
    "user_id",
    "user_session"
])

print(silver_df.count())


In [0]:
from pyspark.sql.functions import col, to_timestamp

silver_df = (
    silver_df
    .withColumn("event_time", to_timestamp("event_time", "yyyy-MM-dd HH:mm:ss 'UTC'"))
    .withColumn("product_id", col("product_id").cast("long"))
    .withColumn("category_id", col("category_id").cast("long"))
    .withColumn("price", col("price").cast("double"))
    .withColumn("user_id", col("user_id").cast("long"))
)

silver_df.printSchema()

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

silver_df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in ["event_time", "product_id", "category_id", "price", "user_id"]
]).show()

In [0]:
silver_df.selectExpr(
    "min(price) as min_price",
    "max(price) as max_price",
    "sum(case when price <= 0 then 1 else 0 end) as non_positive_price_count"
).show()

In [0]:
silver_df = silver_df.filter("price > 0")

print(silver_df.count())

In [0]:
silver_df.write.mode("overwrite").saveAsTable(
    "workspace.silver.ecommerce_events_clean"
)

## 4. Gold Layer - Business Metrics

Create business-ready aggregations for purchase revenue, brand performance, conversion funnel analysis, and product revenue.


In [0]:
purchase_df = silver_df.filter("event_type = 'purchase'")

purchase_df.selectExpr(
    "count(*) as purchase_events",
    "sum(price) as total_revenue"
).show()

In [0]:
from pyspark.sql.functions import avg, count, round, sum

brand_revenue_df = (
    silver_df
    .filter("event_type = 'purchase' AND brand IS NOT NULL")
    .groupBy("brand")
    .agg(
        count("*").alias("purchases"),
        round(sum("price"), 2).alias("revenue"),
        round(avg("price"), 2).alias("avg_purchase_price")
    )
    .orderBy("revenue", ascending=False)
)

display(brand_revenue_df)


In [0]:
brand_revenue_df.write.mode("overwrite").saveAsTable(
    "workspace.gold.brand_revenue"
)

### Session Conversion Funnel

A raw event-count funnel can be misleading because one session may contain multiple events.  
Instead, create one row per user session and track whether that session contained a view, cart, and purchase.


In [0]:
from pyspark.sql.functions import col, max, when

# Create one row per session with flags showing which funnel stages occurred.
session_steps_df = (
    silver_df
    .groupBy("user_session")
    .agg(
        max(when(col("event_type") == "view", 1).otherwise(0)).alias("had_view"),
        max(when(col("event_type") == "cart", 1).otherwise(0)).alias("had_cart"),
        max(when(col("event_type") == "purchase", 1).otherwise(0)).alias("had_purchase")
    )
)

display(session_steps_df)


In [0]:
funnel_counts = session_steps_df.selectExpr(
    "sum(had_view) as view_sessions",
    "sum(case when had_view = 1 and had_cart = 1 then 1 else 0 end) as view_to_cart_sessions",
    "sum(case when had_view = 1 and had_cart = 1 and had_purchase = 1 then 1 else 0 end) as completed_funnel_sessions"
)

display(funnel_counts)

In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import round

# Read the funnel counts dynamically instead of hardcoding values.
counts = funnel_counts.first()

view_sessions = counts["view_sessions"]
view_to_cart_sessions = counts["view_to_cart_sessions"]
completed_funnel_sessions = counts["completed_funnel_sessions"]

view_to_cart_rate = view_to_cart_sessions / view_sessions * 100
cart_to_purchase_rate = completed_funnel_sessions / view_to_cart_sessions * 100
view_to_purchase_rate = completed_funnel_sessions / view_sessions * 100

print(f"View → Cart: {view_to_cart_rate:.2f}%")
print(f"Cart → Purchase: {cart_to_purchase_rate:.2f}%")
print(f"View → Purchase: {view_to_purchase_rate:.2f}%")

funnel_metrics_df = spark.createDataFrame([
    Row(stage="view", sessions=view_sessions, conversion_rate=100.00),
    Row(stage="cart", sessions=view_to_cart_sessions, conversion_rate=view_to_cart_rate),
    Row(stage="purchase", sessions=completed_funnel_sessions, conversion_rate=view_to_purchase_rate)
])

funnel_metrics_df = funnel_metrics_df.withColumn(
    "conversion_rate",
    round("conversion_rate", 2)
)

display(funnel_metrics_df)


In [0]:
funnel_metrics_df.write.mode("overwrite").saveAsTable(
    "workspace.gold.funnel_metrics"
)

In [0]:
from pyspark.sql.functions import sum, count, round

product_revenue_df = (
    silver_df
    .filter("event_type = 'purchase'")
    .groupBy("product_id")
    .agg(
        count("*").alias("purchases"),
        round(sum("price"), 2).alias("revenue")
    )
    .orderBy("revenue", ascending=False)
)

display(product_revenue_df)


In [0]:
product_revenue_df.write.mode("overwrite").saveAsTable(
    "workspace.gold.product_revenue"
)

## 5. Visualizations

Use the Gold tables to visualize top brands by revenue, the session conversion funnel, and top products by revenue.


In [0]:
top_brands_df = spark.table("workspace.gold.brand_revenue").limit(10)

display(top_brands_df)

Databricks visualization. Run in Databricks to view.

In [0]:
funnel_chart_df = spark.table("workspace.gold.funnel_metrics")

display(funnel_chart_df)

Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import col

top_products_chart_df = (
    spark.table("workspace.gold.product_revenue")
    .limit(10)
    .withColumn("product_id", col("product_id").cast("string"))
)

display(top_products_chart_df)

Databricks visualization. Run in Databricks to view.